## GC_Mendelian_variants (n: 221 (174 variants))
- within the variant map, but not in the region file 
- has a vcf file

In [84]:
from importlib import reload
import pandas as pd
import sys
import os
import yaml
sys.path.append('../helpful_functions')
import helpful_functions as hf
reload(hf)

<module 'helpful_functions' from '/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/../helpful_functions/helpful_functions.py'>

In [85]:
# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'
my_col_ref_base = 'tmp_ref_base'
my_col_alt_base = 'tmp_alt_base'


interesting_columns = [col_name, col_sequence, col_category, col_class, col_source,
                       col_ref, col_chr, col_start, col_end, col_strand, col_variant_class, col_variant_pos, col_SPDI, col_allele, col_info]

In [86]:
# config
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/config_file.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

input_fasta = config['design_file']
pre_metadata_df = hf.fasta_to_dataframe(input_fasta, columns=[col_name, col_sequence])
# split the metadata file headers by '#'
# Apply the function to each row and concatenate the results
pre_metadata_df_split = pd.concat(pre_metadata_df.apply(lambda row: hf.split_ids(row, id_col=col_name, separator='#'), axis=1).values)

# Reset the index
pre_metadata_df_split.reset_index(drop=True, inplace=True)

pre_metadata_df = pre_metadata_df_split.copy()
print(pre_metadata_df.shape[0]) # 80803
pre_metadata_df['tmp_label'] = pre_metadata_df[col_name].apply(lambda x: hf.get_label(x))

# variant map
variant_map_path = config['variant_region_map']
variant_map = pd.read_csv(variant_map_path, sep="\t")
# variant_map.columns = ['ID', 'Region', 'REF', 'ALT']
# variant_map.to_csv('/home/kisa/coding/80K_MPRA/design_data/design_info/variant_region_map_new_colnames.tsv.gz', sep="\t", compression='gzip', index=False)
variant_map['tmp_label'] = variant_map['ID'].apply(hf.get_label)

# region bed
region_bed = config['mendelian_regions_bed']
region_bed = pd.read_csv(region_bed, sep="\t", header=None)
region_bed.columns = [f'region_{col_name}' for col_name in ['chr', 'start', 'end', 'name', 'score', 'strand']]

# vcf
vcf_path = config['variant_vcf']
vcf_df = pd.read_csv(vcf_path, sep='\t', comment="#", header=None)
vcf_df.columns = ['CHROM', 'var_pos', 'ID', 'vcf_REF', 'vcf_ALT', 'QUAL', 'FILTER', 'INFO']


80803


In [87]:
variant_map['tmp_label'].value_counts()

tmp_label
cardiac_neuro_cava_random    46374
GC_Selvarajan                  198
GC_Kircher                     198
GC_Mendelian_variants          174
C_positive_heart_CAD            49
GC_Atrial_fib                   23
GC_Mohlke                       20
GC_Liang                         8
Name: count, dtype: int64

In [88]:
group_name = 'GC_Mendelian_variants'
print(f'Number of oligos in the group {group_name}: {pre_metadata_df.loc[pre_metadata_df["tmp_label"] == group_name].shape[0]}')

Number of oligos in the group GC_Mendelian_variants: 221


In [89]:
variant_map.loc[variant_map['tmp_label'] == group_name].copy()

,ID,Region,REF,ALT,tmp_label
46821,GC_Mendelian_variants:chr1:21564170G>A|ALPL,GC_Mendelian_variants:chr1:21564170G>A|ALPL,GC_Mendelian_variants:REF_chr1:21564170G>A|ALPL,GC_Mendelian_variants:ALT_chr1:21564170G>A|ALP...,GC_Mendelian_variants
46822,GC_Mendelian_variants:chr1:209816133C>CA|IRF6,GC_Mendelian_variants:chr1:209816133C>CA|IRF6,GC_Mendelian_variants:REF_chr1:209816133C>CA|IRF6,GC_Mendelian_variants:ALT_chr1:209816133C>CA|I...,GC_Mendelian_variants
46823,GC_Mendelian_variants:chr10:23219376A>C|PTF1A,GC_Mendelian_variants:chr10:23219376A>C|PTF1A,GC_Mendelian_variants:REF_chr10:23219376A>C|PTF1A,GC_Mendelian_variants:ALT_chr10:23219376A>C|PT...,GC_Mendelian_variants
46824,GC_Mendelian_variants:chr10:23219376A>C|PTF1A,GC_Mendelian_variants:chr10:23219434A>G|PTF1A,GC_Mendelian_variants:REF_chr10:23219434A>G|PTF1A,GC_Mendelian_variants:ALT_chr10:23219434A>G|PT...,GC_Mendelian_variants
46825,GC_Mendelian_variants:chr10:23219376A>C|PTF1A,GC_Mendelian_variants:chr10:23219436A>G|PTF1A,GC_Mendelian_variants:REF_chr10:23219436A>G|PTF1A,GC_Mendelian_variants:ALT_chr10:23219436A>G|PT...,GC_Mendelian_variants
...,...,...,...,...,...
46990,GC_Mendelian_variants:chr9:101435912C>T|ALDOB,GC_Mendelian_variants:chr9:101435912C>T|ALDOB,GC_Mendelian_variants:REF_chr9:101435912C>T|ALDOB,GC_Mendelian_variants:ALT_chr9:101435912C>T|AL...,GC_Mendelian_variants
46991,GC_Mendelian_variants:chrX:38352331A>G|OTC,GC_Mendelian_variants:chrX:38352331A>G|OTC,GC_Mendelian_variants:REF_chrX:38352331A>G|OTC,GC_Mendelian_variants:ALT_chrX:38352331A>G|OTC...,GC_Mendelian_variants
46992,GC_Mendelian_variants:chrX:55028202A>G|ALAS2,GC_Mendelian_variants:chrX:55028202A>G|ALAS2,GC_Mendelian_variants:REF_chrX:55028202A>G|ALAS2,GC_Mendelian_variants:ALT_chrX:55028202A>G|ALA...,GC_Mendelian_variants
46993,GC_Mendelian_variants:chrX:55031184G>C|ALAS2,GC_Mendelian_variants:chrX:55031184G>C|ALAS2,GC_Mendelian_variants:REF_chrX:55031184G>C|ALAS2,GC_Mendelian_variants:ALT_chrX:55031184G>C|ALA...,GC_Mendelian_variants


In [90]:
#helpful functions
## Variant / element destinction

def is_variant_related(name, variant_related_list):
    return name in variant_related_list

def is_reference_related(name, reference_related_list):
    return name in reference_related_list

def is_alternative_related(name, variant_related_list):
    return name in variant_related_list

## adding the variant information and combining the columns
# Combine columns
def combine_columns(df):
    new_columns = {}
    for col in df.columns:
        if col.endswith("_x"):
            base_name = col[:-2]  # Remove "_x"
            corresponding_y = base_name + "_y"
            # Combine _x and _y columns
            if corresponding_y in df.columns:
                new_columns[base_name] = df[col].fillna(df[corresponding_y])
            else:
                new_columns[base_name] = df[col]
        elif not col.endswith("_y"):  # Add columns that aren't paired with _x/_y
            new_columns[col] = df[col]
    return pd.DataFrame(new_columns)


## computing more information about the variants
def get_var_pos(row, var_pos_column, allele_column, seq_start_column, variant_1_based=True, start_0_based=True, sequence_length=270):
    """
    Computes the variant pos (0-based coordinate of variant in the string)
    for the alternative sequences with different cased of the data
    (variant_1_based and start_0_based is the default)

    If variant is 0 based set variant_1_based to false (same goes for start)
    """
    variant_position = 'NA'
    if hf.is_alternative(row[allele_column]):
        variant_0_based = row[var_pos_column] - 1 if variant_1_based else row[var_pos_column]
        seq_start_0_based = row[seq_start_column] if start_0_based else row[seq_start_column] - 1
        variant_position = variant_0_based - seq_start_0_based
    if row[col_strand] == '-':
        variant_position = sequence_length - (variant_position + 1) # 0-based variant position
    return variant_position

import re
def find_indel_pattern(row, ref_column, alt_column):
    """Check if in ref or alt is more than 1 subsequent nucleotide indicating an indel
    Special case: I want it to check for the notation of muliallelic variants as well (A,T) if any of these is an indel"""
    if hf.is_alternative(row[col_allele]):
        pattern = r'(^[ACGT]{2,})|(,[ACGT]{2,})'
        # Check if the text matches the pattern
        ref_indel = bool(re.match(pattern, row[ref_column]))
        alt_indel = bool(re.match(pattern, row[alt_column]))
        return ref_indel or alt_indel
    else: False


def get_variant_alternative(row, col_sequence, col_variant_pos, col_allele, col_variant_class='variant_class'):
    """Return the char at the variant pos position"""
    if not hf.is_alternative(row[col_allele]):
        return 'NA'
    if row[col_variant_class] == 'SNV':
        variant_position = int(row[col_variant_pos])
        return row[col_sequence][variant_position]
    elif row[col_variant_class] == 'indel':
        if ',' in row['vcf_ALT']:
            raise ValueError('Special case of indel. Please check manually')
        return row['vcf_ALT']
    return 'NA'

## create SPDI
# dict of chr number to refseq chromosome number
chrom_2_refseq = {"chr1": "NC_000001.11",
    "chr2": "NC_000002.12",
    "chr3": "NC_000003.12",
    "chr4": "NC_000004.12",
    "chr5": "NC_000005.10",
    "chr6": "NC_000006.12",
    "chr7": "NC_000007.14",
    "chr8": "NC_000008.11",
    "chr9": "NC_000009.12",
    "chr10": "NC_000010.11",
    "chr11": "NC_000011.10",
    "chr12": "NC_000012.12",
    "chr13": "NC_000013.11",
    "chr14": "NC_000014.9",
    "chr15": "NC_000015.10",
    "chr16": "NC_000016.10",
    "chr17": "NC_000017.11",
    "chr18": "NC_000018.10",
    "chr19": "NC_000019.10",
    "chr20": "NC_000020.11",
    "chr21": "NC_000021.9",
    "chr22": "NC_000022.11",
    "chrX": "NC_000023.11",
    "chrY": "NC_000024.10"}


def create_speedy_chromosomes(user_string, seperator="-", indices=[0,1,2,3]):
    """
    Returns SPDI identifier for given variant
    spdi: refseq_chromosome:pos:ref:alt
    @params: indices: list of indices for chromosome, position, ref and alt in user_string
    """
    if len(user_string.split(seperator)) < 4:
        raise ValueError('Not enough indices given: Expected chrom, pos, ref, alt position in user_string')
    split_string = user_string.split(seperator)
    zero_based_position = int(split_string[indices[1]]) - 1 # (input: 1-based => 0-based)
    return f'{chrom_2_refseq[split_string[indices[0]]]}:{zero_based_position}:{split_string[indices[2]]}:{split_string[indices[3]]}'

In [91]:
region_bed['region_name_label'] = region_bed['region_name'].apply(lambda name: f'{group_name}:{name}')

In [92]:
region_bed

,region_chr,region_start,region_end,region_name,region_score,region_strand,region_name_label
0,chr1,7961723,7961993,chr1:7961859C>G|PARK7,.,.,GC_Mendelian_variants:chr1:7961859C>G|PARK7
1,chr1,21564034,21564304,chr1:21564170G>A|ALPL,.,.,GC_Mendelian_variants:chr1:21564170G>A|ALPL
2,chr1,155301331,155301601,chr1:155301467T>C|PKLR,.,.,GC_Mendelian_variants:chr1:155301467T>C|PKLR
3,chr1,155301342,155301612,chr1:155301478C>G|PKLR,.,.,GC_Mendelian_variants:chr1:155301478C>G|PKLR
4,chr1,160031873,160032143,chr1:160032009G>C|PIGM,.,.,GC_Mendelian_variants:chr1:160032009G>C|PIGM
...,...,...,...,...,...,...,...
179,chrX,155022634,155022904,chrX:155022770AG>G|F8,.,.,GC_Mendelian_variants:chrX:155022770AG>G|F8
180,chrX,155022637,155022907,chrX:155022773A>T|F8,.,.,GC_Mendelian_variants:chrX:155022773A>T|F8
181,chrX,155022671,155022941,chrX:155022807T>C|F8,.,.,GC_Mendelian_variants:chrX:155022807T>C|F8
182,chrX,155022673,155022943,chrX:155022809A>C|F8,.,.,GC_Mendelian_variants:chrX:155022809A>C|F8


In [93]:
variant_region_map = variant_map.merge(region_bed, left_on='Region', right_on='region_name_label', how='inner')
variant_region_map

,ID,Region,REF,ALT,tmp_label,region_chr,region_start,region_end,region_name,region_score,region_strand,region_name_label
0,GC_Mendelian_variants:chr1:21564170G>A|ALPL,GC_Mendelian_variants:chr1:21564170G>A|ALPL,GC_Mendelian_variants:REF_chr1:21564170G>A|ALPL,GC_Mendelian_variants:ALT_chr1:21564170G>A|ALP...,GC_Mendelian_variants,chr1,21564034,21564304,chr1:21564170G>A|ALPL,.,.,GC_Mendelian_variants:chr1:21564170G>A|ALPL
1,GC_Mendelian_variants:chr1:209816133C>CA|IRF6,GC_Mendelian_variants:chr1:209816133C>CA|IRF6,GC_Mendelian_variants:REF_chr1:209816133C>CA|IRF6,GC_Mendelian_variants:ALT_chr1:209816133C>CA|I...,GC_Mendelian_variants,chr1,209815997,209816267,chr1:209816133C>CA|IRF6,.,.,GC_Mendelian_variants:chr1:209816133C>CA|IRF6
2,GC_Mendelian_variants:chr10:23219376A>C|PTF1A,GC_Mendelian_variants:chr10:23219376A>C|PTF1A,GC_Mendelian_variants:REF_chr10:23219376A>C|PTF1A,GC_Mendelian_variants:ALT_chr10:23219376A>C|PT...,GC_Mendelian_variants,chr10,23219240,23219510,chr10:23219376A>C|PTF1A,.,.,GC_Mendelian_variants:chr10:23219376A>C|PTF1A
3,GC_Mendelian_variants:chr10:23219376A>C|PTF1A,GC_Mendelian_variants:chr10:23219434A>G|PTF1A,GC_Mendelian_variants:REF_chr10:23219434A>G|PTF1A,GC_Mendelian_variants:ALT_chr10:23219434A>G|PT...,GC_Mendelian_variants,chr10,23219298,23219568,chr10:23219434A>G|PTF1A,.,.,GC_Mendelian_variants:chr10:23219434A>G|PTF1A
4,GC_Mendelian_variants:chr10:23219376A>C|PTF1A,GC_Mendelian_variants:chr10:23219436A>G|PTF1A,GC_Mendelian_variants:REF_chr10:23219436A>G|PTF1A,GC_Mendelian_variants:ALT_chr10:23219436A>G|PT...,GC_Mendelian_variants,chr10,23219300,23219570,chr10:23219436A>G|PTF1A,.,.,GC_Mendelian_variants:chr10:23219436A>G|PTF1A
...,...,...,...,...,...,...,...,...,...,...,...,...
169,GC_Mendelian_variants:chr9:101435912C>T|ALDOB,GC_Mendelian_variants:chr9:101435912C>T|ALDOB,GC_Mendelian_variants:REF_chr9:101435912C>T|ALDOB,GC_Mendelian_variants:ALT_chr9:101435912C>T|AL...,GC_Mendelian_variants,chr9,101435776,101436046,chr9:101435912C>T|ALDOB,.,.,GC_Mendelian_variants:chr9:101435912C>T|ALDOB
170,GC_Mendelian_variants:chrX:38352331A>G|OTC,GC_Mendelian_variants:chrX:38352331A>G|OTC,GC_Mendelian_variants:REF_chrX:38352331A>G|OTC,GC_Mendelian_variants:ALT_chrX:38352331A>G|OTC...,GC_Mendelian_variants,chrX,38352195,38352465,chrX:38352331A>G|OTC,.,.,GC_Mendelian_variants:chrX:38352331A>G|OTC
171,GC_Mendelian_variants:chrX:55028202A>G|ALAS2,GC_Mendelian_variants:chrX:55028202A>G|ALAS2,GC_Mendelian_variants:REF_chrX:55028202A>G|ALAS2,GC_Mendelian_variants:ALT_chrX:55028202A>G|ALA...,GC_Mendelian_variants,chrX,55028066,55028336,chrX:55028202A>G|ALAS2,.,.,GC_Mendelian_variants:chrX:55028202A>G|ALAS2
172,GC_Mendelian_variants:chrX:55031184G>C|ALAS2,GC_Mendelian_variants:chrX:55031184G>C|ALAS2,GC_Mendelian_variants:REF_chrX:55031184G>C|ALAS2,GC_Mendelian_variants:ALT_chrX:55031184G>C|ALA...,GC_Mendelian_variants,chrX,55031048,55031318,chrX:55031184G>C|ALAS2,.,.,GC_Mendelian_variants:chrX:55031184G>C|ALAS2


In [94]:
pre_metadata_df_group = pre_metadata_df.loc[pre_metadata_df['tmp_label'] == group_name].copy()
expected_number = pre_metadata_df_group.shape[0]
variant_region_df_group = variant_region_map.loc[variant_region_map['tmp_label'] == group_name].copy()
region_bed_group = region_bed.loc[region_bed['region_name'].str.contains(group_name)].copy()
vcf_df_group = vcf_df.loc[vcf_df['ID'].str.contains(group_name)].copy()

In [95]:
# add the columns of the metadata file
pre_metadata_df_group[col_sequence] = pre_metadata_df_group[col_sequence].apply(lambda x: x[15:-15])

# if variant related or element
variant_related_list = set(variant_region_df_group['REF'].to_list()).union(set(variant_region_df_group['ALT'].to_list()))
pre_metadata_df_group[col_category] = pre_metadata_df_group[col_name].apply(lambda name: 'variant' if is_variant_related(name, variant_related_list) else 'element')


pre_metadata_df_group[col_class] = pre_metadata_df_group[col_name].apply(lambda name: 'variant negative control' if is_variant_related(name, variant_related_list) else 'element inactive control')
pre_metadata_df_group[col_source] = 'general controls IGVF year 1 design 2023'
pre_metadata_df_group[col_ref] = 'GRCh38'

# add allele
alternative_related_list = set(variant_region_df_group['ALT'].to_list())
reference_related_list = set(variant_region_df_group['REF'].to_list())
pre_metadata_df_group[col_allele] = pre_metadata_df_group[col_name].apply(lambda name: 'alt' if is_alternative_related(name, alternative_related_list) else 'ref' if is_reference_related(name, reference_related_list) else 'NA')

pre_metadata_df_group[col_variant_class] = 'NA'
pre_metadata_df_group[col_variant_pos] = 'NA'
pre_metadata_df_group[col_SPDI] = 'NA'
pre_metadata_df_group[col_info] = '' # 'Coordinates are based on GRCh37 (wrong genome build)'

### Focus on Elements

In [96]:
pre_metadata_df_group_element = pre_metadata_df_group.loc[pre_metadata_df_group[col_category] == 'element']
pre_metadata_df_group_element.name.to_list()

[]

### Focus on Variants

In [97]:
pre_metadata_df_group_variant = pre_metadata_df_group.loc[pre_metadata_df_group[col_category] == 'variant'].copy()
print(pre_metadata_df_group_variant.shape[0])
pre_metadata_df_group_variant[col_name].nunique()

221


221

In [98]:
print(f'Expected variant number of this group: {variant_region_df_group.shape[0]}')

Expected variant number of this group: 174


In [99]:
vcf_df_group.loc[vcf_df_group['ID'].str.contains('GC_Mendelian_variants:chr7:156791274T>TTAAGGAA')]['ID'].to_list()

['GC_Mendelian_variants:chr7:156791274T>TTAAGGAAGTGATT|SHH']

In [100]:
variant_region_vcf_group = variant_region_df_group.merge(vcf_df_group, on='ID', how='inner')

In [101]:
variant_region_vcf_group

,ID,Region,REF,ALT,tmp_label,region_chr,region_start,region_end,region_name,region_score,region_strand,region_name_label,CHROM,var_pos,vcf_REF,vcf_ALT,QUAL,FILTER,INFO
0,GC_Mendelian_variants:chr1:21564170G>A|ALPL,GC_Mendelian_variants:chr1:21564170G>A|ALPL,GC_Mendelian_variants:REF_chr1:21564170G>A|ALPL,GC_Mendelian_variants:ALT_chr1:21564170G>A|ALP...,GC_Mendelian_variants,chr1,21564034,21564304,chr1:21564170G>A|ALPL,.,.,GC_Mendelian_variants:chr1:21564170G>A|ALPL,chr1,21564170,G,A,.,PASS,gene=ALPL;PMID=10679946;Enhancer;Region=chr1:2...
1,GC_Mendelian_variants:chr1:209816133C>CA|IRF6,GC_Mendelian_variants:chr1:209816133C>CA|IRF6,GC_Mendelian_variants:REF_chr1:209816133C>CA|IRF6,GC_Mendelian_variants:ALT_chr1:209816133C>CA|I...,GC_Mendelian_variants,chr1,209815997,209816267,chr1:209816133C>CA|IRF6,.,.,GC_Mendelian_variants:chr1:209816133C>CA|IRF6,chr1,209816133,C,CA,.,PASS,gene=IRF6;PMID=24442519;Enhancer;Region=chr1:2...
2,GC_Mendelian_variants:chr10:23219376A>C|PTF1A,GC_Mendelian_variants:chr10:23219376A>C|PTF1A,GC_Mendelian_variants:REF_chr10:23219376A>C|PTF1A,GC_Mendelian_variants:ALT_chr10:23219376A>C|PT...,GC_Mendelian_variants,chr10,23219240,23219510,chr10:23219376A>C|PTF1A,.,.,GC_Mendelian_variants:chr10:23219376A>C|PTF1A,chr10,23219376,A,C,.,PASS,gene=PTF1A;PMID=24212882;Enhancer;Region=chr10...
3,GC_Mendelian_variants:chr10:23219376A>C|PTF1A,GC_Mendelian_variants:chr10:23219434A>G|PTF1A,GC_Mendelian_variants:REF_chr10:23219434A>G|PTF1A,GC_Mendelian_variants:ALT_chr10:23219434A>G|PT...,GC_Mendelian_variants,chr10,23219298,23219568,chr10:23219434A>G|PTF1A,.,.,GC_Mendelian_variants:chr10:23219434A>G|PTF1A,chr10,23219376,A,C,.,PASS,gene=PTF1A;PMID=24212882;Enhancer;Region=chr10...
4,GC_Mendelian_variants:chr10:23219376A>C|PTF1A,GC_Mendelian_variants:chr10:23219436A>G|PTF1A,GC_Mendelian_variants:REF_chr10:23219436A>G|PTF1A,GC_Mendelian_variants:ALT_chr10:23219436A>G|PT...,GC_Mendelian_variants,chr10,23219300,23219570,chr10:23219436A>G|PTF1A,.,.,GC_Mendelian_variants:chr10:23219436A>G|PTF1A,chr10,23219376,A,C,.,PASS,gene=PTF1A;PMID=24212882;Enhancer;Region=chr10...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169,GC_Mendelian_variants:chr9:101435912C>T|ALDOB,GC_Mendelian_variants:chr9:101435912C>T|ALDOB,GC_Mendelian_variants:REF_chr9:101435912C>T|ALDOB,GC_Mendelian_variants:ALT_chr9:101435912C>T|AL...,GC_Mendelian_variants,chr9,101435776,101436046,chr9:101435912C>T|ALDOB,.,.,GC_Mendelian_variants:chr9:101435912C>T|ALDOB,chr9,101435912,C,T,.,PASS,gene=ALDOB;PMID=20882353;Promoter;Region=chr9:...
170,GC_Mendelian_variants:chrX:38352331A>G|OTC,GC_Mendelian_variants:chrX:38352331A>G|OTC,GC_Mendelian_variants:REF_chrX:38352331A>G|OTC,GC_Mendelian_variants:ALT_chrX:38352331A>G|OTC...,GC_Mendelian_variants,chrX,38352195,38352465,chrX:38352331A>G|OTC,.,.,GC_Mendelian_variants:chrX:38352331A>G|OTC,chrX,38352331,A,G,.,PASS,gene=OTC;PMID=20127982;Promoter;Region=chrX:38...
171,GC_Mendelian_variants:chrX:55028202A>G|ALAS2,GC_Mendelian_variants:chrX:55028202A>G|ALAS2,GC_Mendelian_variants:REF_chrX:55028202A>G|ALAS2,GC_Mendelian_variants:ALT_chrX:55028202A>G|ALA...,GC_Mendelian_variants,chrX,55028066,55028336,chrX:55028202A>G|ALAS2,.,.,GC_Mendelian_variants:chrX:55028202A>G|ALAS2,chrX,55028202,A,G,.,PASS,gene=ALAS2;PMID=23935018;Enhancer;Region=chrX:...
172,GC_Mendelian_variants:chrX:55031184G>C|ALAS2,GC_Mendelian_variants:chrX:55031184G>C|ALAS2,GC_Mendelian_variants:REF_chrX:55031184G>C|ALAS2,GC_Mendelian_variants:ALT_chrX:55031184G>C|ALA...,GC_Mendelian_variants,chrX,55031048,55031318,chrX:55031184G>C|ALAS2,.,.,GC_Mendelian_variants:chrX:55031184G>C|ALAS2,chrX,55031184,G,C,.,PASS,gene=ALAS2;PMID=12663458;Promoter;Region=chrX:...


#### Match this information to the sequences

In [102]:
only_reference_sequences = variant_region_vcf_group[['REF', 'region_chr', 'region_start', 'region_end', 'region_strand']].drop_duplicates(subset=['REF', 'region_chr', 'region_start', 'region_end', 'region_strand'])
pre_metadata_df_group_region_ref = pre_metadata_df_group_variant.merge(only_reference_sequences, left_on=col_name, right_on='REF', how='left')
print(pre_metadata_df_group_region_ref.shape[0])
print(pre_metadata_df_group_region_ref[col_name].nunique())
pre_metadata_df_group_region_ref_alt = pre_metadata_df_group_region_ref.merge(variant_region_vcf_group[['ALT', 'region_chr', 'region_start', 'region_end', 'region_strand', 'var_pos', 'vcf_REF', 'vcf_ALT']], left_on=col_name, right_on='ALT', how='left')
print(pre_metadata_df_group_region_ref_alt.shape[0])
print(pre_metadata_df_group_region_ref_alt[col_name].nunique())

221
221
221
221


##### Combine x and y columns

In [103]:
pre_metadata_df_group_region_ref_alt = combine_columns(pre_metadata_df_group_region_ref_alt)

In [104]:
pre_metadata_df_group_region_ref_alt
pre_metadata_df_group_region_ref_alt.rename(columns=lambda x: x.replace('region_', '') if 'region_' in x else x , inplace=True)
# Converting float columns to integers
pre_metadata_df_group_region_ref_alt['start'] = pre_metadata_df_group_region_ref_alt['start'].astype(int)
pre_metadata_df_group_region_ref_alt['end'] = pre_metadata_df_group_region_ref_alt['end'].astype(int)

#### Compute variant position
- is the variant position 1-based - yes (5254956) https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr11%3A5254955%2D5254957&hgsid=2392258313_huE5stAQaQkB4376zP5rx9IaAEMn
- start 0-based: chr16:159,574-159,584 
https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr16%3A159574%2D159584&hgsid=2392258313_huE5stAQaQkB4376zP5rx9IaAEMn

In [105]:
pre_metadata_df_group_region_ref_alt

,name,sequence,tmp_label,category,class,source,ref,allele,variant_class,variant_pos,...,info,REF,chr,start,end,strand,ALT,var_pos,vcf_REF,vcf_ALT
0,GC_Mendelian_variants:REF_chr1:21564170G>A|ALPL,CCCCAGGGAAATCTGTGGGCATTGTGACCACCACGAGAGTGAACCA...,GC_Mendelian_variants,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,...,,GC_Mendelian_variants:REF_chr1:21564170G>A|ALPL,chr1,21564034,21564304,.,NaN,NaN,NaN,NaN
1,GC_Mendelian_variants:REF_chr1:209816133C>CA|IRF6,TTGAGCCCAGGGGCTGAATCTGGAGCTTTGGGGCCTGGGAACCTCT...,GC_Mendelian_variants,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,...,,GC_Mendelian_variants:REF_chr1:209816133C>CA|IRF6,chr1,209815997,209816267,.,NaN,NaN,NaN,NaN
2,GC_Mendelian_variants:REF_chr10:23219376A>C|PTF1A,ATTTGGGTTTCTCCTGTGTTTCAGATACTGATGTTTGAGCTTTCTC...,GC_Mendelian_variants,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,...,,GC_Mendelian_variants:REF_chr10:23219376A>C|PTF1A,chr10,23219240,23219510,.,NaN,NaN,NaN,NaN
3,GC_Mendelian_variants:REF_chr10:23219434A>G|PTF1A,CACTTAAAGAGTCACTGTTACTTTGAGGTTTTATCTGTAAGATTCG...,GC_Mendelian_variants,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,...,,GC_Mendelian_variants:REF_chr10:23219434A>G|PTF1A,chr10,23219298,23219568,.,NaN,NaN,NaN,NaN
4,GC_Mendelian_variants:REF_chr10:23219436A>G|PTF1A,CTTAAAGAGTCACTGTTACTTTGAGGTTTTATCTGTAAGATTCGTG...,GC_Mendelian_variants,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,...,,GC_Mendelian_variants:REF_chr10:23219436A>G|PTF1A,chr10,23219300,23219570,.,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
216,GC_Mendelian_variants:ALT_chr9:101435912C>T|AL...,AGATCTTTGGTAGCACACAATTTTTATAGACTTCTCATCATGTTTT...,GC_Mendelian_variants,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,alt,NA,NA,...,,NaN,chr9,101435776,101436046,.,GC_Mendelian_variants:ALT_chr9:101435912C>T|AL...,101435912.0,C,T
217,GC_Mendelian_variants:ALT_chrX:38352331A>G|OTC...,GTGGAAAGACTGGCAATTAGAGGTAGAAAAGTGAAATAAATGGAAA...,GC_Mendelian_variants,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,alt,NA,NA,...,,NaN,chrX,38352195,38352465,.,GC_Mendelian_variants:ALT_chrX:38352331A>G|OTC...,38352331.0,A,G
218,GC_Mendelian_variants:ALT_chrX:55028202A>G|ALA...,AGGTAATTATGGCCACTTCACAAGGTAGGTAAGGAATTGGATCCAG...,GC_Mendelian_variants,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,alt,NA,NA,...,,NaN,chrX,55028066,55028336,.,GC_Mendelian_variants:ALT_chrX:55028202A>G|ALA...,55028202.0,A,G
219,GC_Mendelian_variants:ALT_chrX:55031184G>C|ALA...,GGCCTGGCCCTGCATTGGCCCCAAAGGTATCTCAGTCCCTTCCTTG...,GC_Mendelian_variants,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,alt,NA,NA,...,,NaN,chrX,55031048,55031318,.,GC_Mendelian_variants:ALT_chrX:55031184G>C|ALA...,55031184.0,G,C


In [106]:
pre_metadata_df_group_region_ref_alt["is_indel"] = pre_metadata_df_group_region_ref_alt.apply(lambda row: find_indel_pattern(row, "vcf_REF", "vcf_ALT"), axis=1)
# add variant_class: SNV or indel
pre_metadata_df_group_region_ref_alt[col_variant_class] = pre_metadata_df_group_region_ref_alt['is_indel'].apply(lambda indel: 'indel' if indel else 'SNV')

pre_metadata_df_group_region_ref_alt[col_variant_pos] = pre_metadata_df_group_region_ref_alt.apply(lambda row: get_var_pos(row, var_pos_column='var_pos', allele_column=col_allele, seq_start_column=col_start, variant_1_based=True, start_0_based=True), axis=1)
pre_metadata_df_group_region_ref_alt[[col_sequence, 'var_pos', col_chr, col_start, col_end, 'variant_pos', col_allele]]


pre_metadata_df_group_region_ref_alt[[col_name, col_sequence, 'var_pos', col_chr, col_start, col_end, 'variant_pos', col_allele, 'vcf_REF', 'vcf_ALT', col_variant_class]]
pre_metadata_df_group_region_ref_alt[col_variant_class].value_counts()

pre_metadata_df_group_region_ref_alt['real_ALT'] = pre_metadata_df_group_region_ref_alt.apply(lambda row: get_variant_alternative(row, col_sequence=col_sequence, col_variant_pos=col_variant_pos, col_allele=col_allele), axis=1)
pre_metadata_df_group_region_ref_alt[pre_metadata_df_group_region_ref_alt[col_variant_class] == 'SNV'][[col_chr, 'var_pos', 'vcf_REF', 'vcf_ALT', col_name, col_sequence, col_chr, col_start, col_end, col_strand, 'variant_pos', col_allele,  col_variant_class, 'real_ALT']]


,chr,var_pos,vcf_REF,vcf_ALT,name,sequence,chr,start,end,strand,variant_pos,allele,variant_class,real_ALT
0,chr1,NaN,NaN,NaN,GC_Mendelian_variants:REF_chr1:21564170G>A|ALPL,CCCCAGGGAAATCTGTGGGCATTGTGACCACCACGAGAGTGAACCA...,chr1,21564034,21564304,.,NA,ref,SNV,NA
1,chr1,NaN,NaN,NaN,GC_Mendelian_variants:REF_chr1:209816133C>CA|IRF6,TTGAGCCCAGGGGCTGAATCTGGAGCTTTGGGGCCTGGGAACCTCT...,chr1,209815997,209816267,.,NA,ref,SNV,NA
2,chr10,NaN,NaN,NaN,GC_Mendelian_variants:REF_chr10:23219376A>C|PTF1A,ATTTGGGTTTCTCCTGTGTTTCAGATACTGATGTTTGAGCTTTCTC...,chr10,23219240,23219510,.,NA,ref,SNV,NA
3,chr10,NaN,NaN,NaN,GC_Mendelian_variants:REF_chr10:23219434A>G|PTF1A,CACTTAAAGAGTCACTGTTACTTTGAGGTTTTATCTGTAAGATTCG...,chr10,23219298,23219568,.,NA,ref,SNV,NA
4,chr10,NaN,NaN,NaN,GC_Mendelian_variants:REF_chr10:23219436A>G|PTF1A,CTTAAAGAGTCACTGTTACTTTGAGGTTTTATCTGTAAGATTCGTG...,chr10,23219300,23219570,.,NA,ref,SNV,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
216,chr9,101435912.0,C,T,GC_Mendelian_variants:ALT_chr9:101435912C>T|AL...,AGATCTTTGGTAGCACACAATTTTTATAGACTTCTCATCATGTTTT...,chr9,101435776,101436046,.,135.0,alt,SNV,T
217,chrX,38352331.0,A,G,GC_Mendelian_variants:ALT_chrX:38352331A>G|OTC...,GTGGAAAGACTGGCAATTAGAGGTAGAAAAGTGAAATAAATGGAAA...,chrX,38352195,38352465,.,135.0,alt,SNV,G
218,chrX,55028202.0,A,G,GC_Mendelian_variants:ALT_chrX:55028202A>G|ALA...,AGGTAATTATGGCCACTTCACAAGGTAGGTAAGGAATTGGATCCAG...,chrX,55028066,55028336,.,135.0,alt,SNV,G
219,chrX,55031184.0,G,C,GC_Mendelian_variants:ALT_chrX:55031184G>C|ALA...,GGCCTGGCCCTGCATTGGCCCCAAAGGTATCTCAGTCCCTTCCTTG...,chrX,55031048,55031318,.,135.0,alt,SNV,C


### Add SPDI

In [107]:
# add SPDI for alt
pre_metadata_df_group_region_ref_alt['SPDI'] = pre_metadata_df_group_region_ref_alt.apply(lambda row: create_speedy_chromosomes(f"{row['chr']}-{int(row['var_pos'])}-{row['vcf_REF']}-{row['real_ALT']}") if hf.is_alternative(row[col_allele]) else 'NA', axis=1)

# add SPDI for ref
# make dict for REF: [list of ALT_IDs associated to this REF] from the variant_map_filtered
ref_alt_dict = variant_region_df_group.groupby('REF')['ALT'].apply(list).to_dict()
ref_alt_dict

# # make dict for ALT_ID to SPDI from the metadata table
alt_spdi_dict = pre_metadata_df_group_region_ref_alt.loc[pre_metadata_df_group_region_ref_alt[col_allele].apply(hf.is_alternative)][[col_name, col_SPDI]].set_index(col_name).to_dict()[col_SPDI]
alt_spdi_dict

# # make dict for ALT_ID to variant_pos from the metadata table
alt_variant_pos_dict = pre_metadata_df_group_region_ref_alt.loc[pre_metadata_df_group_region_ref_alt[col_allele].apply(hf.is_alternative)][[col_name, col_variant_pos]].set_index(col_name).to_dict()[col_variant_pos]
alt_variant_pos_dict

# # make dict for ALT_ID to variant_class from the metadata table
alt_variant_class_dict = pre_metadata_df_group_region_ref_alt.loc[pre_metadata_df_group_region_ref_alt[col_allele].apply(hf.is_alternative)][[col_name, col_variant_class]].set_index(col_name).to_dict()[col_variant_class]
alt_variant_class_dict

# function to add a list of SPDI values from the REF to the metadata table
def add_spdi_values_2_reference(row):
    if hf.is_reference(row[col_allele]):
        # check if row[col_name] is in ref_alt_dict
        if not row[col_name] in ref_alt_dict:
            # raise exception
            raise ValueError('Reference ID not found in ref_alt_dict')
        row[col_SPDI] = [alt_spdi_dict[alt_id] for alt_id in ref_alt_dict[row[col_name]]]
        # NOTE: within variant_pos: for reference sequences a array of variant positions need to be added
        row[col_variant_pos] = [int(alt_variant_pos_dict[alt_id]) for alt_id in ref_alt_dict[row[col_name]]]
        row[col_variant_class] = [alt_variant_class_dict[alt_id] for alt_id in ref_alt_dict[row[col_name]]]
        row[col_allele] = ['ref' for _ in ref_alt_dict[row[col_name]]]
    return row

# function to generate arrays out off the columns
def make_column_arrays(row):
    """
    create arrays for the required columns
    """
    allele = row[col_allele]
    SPDI = row[col_SPDI]
    variant_pos = row[col_variant_pos]
    variant_class = row[col_variant_class]

    if allele == 'ref' or allele == 'alt': # only "alt" is string
        row[col_allele] = [allele]
    if isinstance(SPDI, str): # only for alt sequences this is true
        if SPDI != "NA":
            row[col_SPDI] = [SPDI]
    if isinstance(variant_pos, float):
        row[col_variant_pos] = [int(variant_pos)]
    elif isinstance(variant_pos, int):
        row[col_variant_pos] = [int(variant_pos)]
    if isinstance(variant_class, str):
        if row[col_variant_class] in ['SNV', 'indel']:
                row[col_variant_class] = [variant_class]
    if not isinstance(row[col_class], str):
        print(row[col_name])
    return row

pre_metadata_df_group_region_ref_alt = pre_metadata_df_group_region_ref_alt.apply(add_spdi_values_2_reference, axis=1)
pre_metadata_df_group_region_ref_alt = pre_metadata_df_group_region_ref_alt.apply(make_column_arrays, axis = 1)


In [108]:
print('expected number of rows within metadata file:', expected_number)
print('Number of rows in metadata file:', pre_metadata_df_group_region_ref_alt.shape[0])

expected number of rows within metadata file: 221
Number of rows in metadata file: 221


In [109]:
pre_metadata_df_group_region_ref_alt.columns

Index(['name', 'sequence', 'tmp_label', 'category', 'class', 'source', 'ref',
       'allele', 'variant_class', 'variant_pos', 'SPDI', 'info', 'REF', 'chr',
       'start', 'end', 'strand', 'ALT', 'var_pos', 'vcf_REF', 'vcf_ALT',
       'is_indel', 'real_ALT'],
      dtype='object')

In [110]:
# no duplicates in the file: (needed to remove SPDI and allele because list elements in these columns)
pre_metadata_df_group_region_ref_alt.duplicated(subset=['name',
 'sequence',
 'category',
 'class',
 'source',
 'ref',
 'chr',
 'start',
 'end',
 'strand',
 'info']).sum()

0

In [111]:
output_dir = config['final_output_dir']
output_path = os.path.join(output_dir, group_name)
# Write DataFrame to TSV file
pre_metadata_df_group_region_ref_alt[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')

0